In [29]:
import h5py
import numpy as np
import pandas as pd
from pathlib import Path

In [30]:
# Input DREAM.3D file and desired CSV name
input_path = Path("E:/Users/lagon/OneDrive - The University of Manchester/Laura Chat/Simulations/c-axis_misorientation_analysis/small microstructure/D3D_Lunty_avec_tickle_c-axis_misalignments.dream3d") 
output_csv = Path("neighbour_angles.csv") 

path = "/DataContainers/ImageDataContainer/CellFeatureData"
neighbour_list = f"{path}/NeighborList"
neighbours = f"{path}/NumNeighbors"             
misalignments = f"{path}/CAxisMisalignmentList"   
avg_misalignments = f"{path}/AvgCAxisMisalignments" 
shared_surface = f"{path}/SharedSurfaceAreaList"     


In [31]:
def reconstruct_ragged(concat, sizes):
    out = []
    idx = 0
    for s in sizes:
        s = int(s)
        out.append(concat[idx: idx + s])
        idx += s
    return out

with h5py.File(input_path, 'r') as h5:
    neigh_concat  = h5[neighbour_list][:]
    sizes         = h5[neighbours][:].astype(np.int64)
    angles_concat = h5[misalignments][:]

    # length = numFeatures
    avg_angles_per_feature = h5[avg_misalignments][:]

    try:
        area_concat = h5[shared_surface][:]
        has_area = True
    except KeyError:
        area_concat = None
        has_area = False

neighbors = reconstruct_ragged(neigh_concat, sizes)
angles    = reconstruct_ragged(angles_concat, sizes)
areas     = reconstruct_ragged(area_concat, sizes) if has_area else None

rows = []
for A in range(1, len(neighbors)):  # skip feature 0
    nlist = neighbors[A]
    alist = angles[A]
    m = len(nlist)

    avg_theta_A = float(avg_angles_per_feature[A])  
    aareas = areas[A] if has_area else None

    for k in range(m):
        B = int(nlist[k])
        if B <= 0 or B == A:
            continue

        theta = float(alist[k])
        areaA = float(aareas[k]) if has_area else float("nan")

        rows.append((A, B, theta, avg_theta_A, areaA))

df_owner = pd.DataFrame(
    rows,
    columns=["FeatureA", "FeatureB", "Angle_deg", "Avg_angle_deg", "SharedArea_um2"]
).sort_values(["FeatureA", "FeatureB"]).reset_index(drop=True)

display(df_owner.head(50))
df_owner.to_csv(output_csv, index=False)
print("Wrote:", output_csv, "rows:", len(df_owner))


C:\Users\lagon\AppData\Local\Temp\ipykernel_10836\400477841.py:5: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  s = int(s)
C:\Users\lagon\AppData\Local\Temp\ipykernel_10836\400477841.py:35: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  avg_theta_A = float(avg_angles_per_feature[A])


,FeatureA,FeatureB,Angle_deg,Avg_angle_deg,SharedArea_um2
0,1,2,67.718842,46.210022,368.463593
1,1,5,21.655010,46.210022,217.349991
2,1,6,14.357546,46.210022,88.966789
3,1,7,68.067894,46.210022,174.488388
4,1,13,20.101751,46.210022,109.439995
5,1,15,71.131882,46.210022,261.885590
6,1,16,60.437248,46.210022,64.742393
7,2,1,67.718842,47.496212,368.463593
8,2,4,55.440975,47.496212,12.617999
9,2,5,49.530209,47.496212,223.966797


Wrote: neighbour_angles.csv rows: 90
